In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch


In [ ]:

def compute_dominant_frequency(window, fs=256):
    """Compute Dominant Frequency for all EEG channels in a given window."""
    dominant_frequencies = []
    
    for i in range(window.shape[1]):  # Iterate over EEG channels
        f, Pxx = welch(window[:, i], fs=fs, nperseg=fs * 2, window='hann', scaling='density')
        
        # Find the frequency with the highest power (dominant frequency) within 1-50 Hz
        valid_range = (f >= 1) & (f <= 50)
        if np.any(valid_range):
            dominant_freq = f[valid_range][np.argmax(Pxx[valid_range])]
        else:
            dominant_freq = np.nan  # If no valid frequencies found
        
        dominant_frequencies.append(dominant_freq)
    
    return dominant_frequencies


In [ ]:

def sliding_window_dominant_frequency(eeg_data, outcomes, eeg_columns, window_size, step_size, fs=256):
    """Extract Dominant Frequency features using a sliding window approach."""
    all_dominant_freq_features = []
    targets = []
    n_samples = eeg_data.shape[0]
    
    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size
        window = eeg_data[start:end]
        outcome_window = outcomes[start:end]

        # Compute Dominant Frequency features for this window
        dominant_freq_values = compute_dominant_frequency(window, fs)

        all_dominant_freq_features.append(dominant_freq_values)
        targets.append(1 if np.any(outcome_window) else 0)  # Assign target based on Outcome
    
    return np.array(all_dominant_freq_features), np.array(targets)


In [ ]:

# Main processing
if __name__ == "__main__":
    # Load EEG data
    eeg_data_path = '/Users/puchku-home/Study/PROJECT/EEG/EEG Assets/chbmit_preprocessed_data.csv' 
    data = pd.read_csv(eeg_data_path)
    eeg_columns = [col for col in data.columns if col != 'Outcome']

    # Convert to NumPy arrays
    eeg_data = np.asarray(data[eeg_columns].values, dtype=np.float32)
    outcomes = np.asarray(data['Outcome'].values, dtype=np.float32)

    # Windowing parameters
    fs = 256  # Sampling frequency
    window_size = fs * 1  # 1-second windows
    step_size = window_size // 2  # 50% overlap

    # Compute Dominant Frequency features
    dominant_freq_features, targets = sliding_window_dominant_frequency(eeg_data, outcomes, eeg_columns, window_size, step_size, fs)

    # Convert to DataFrame and save
    dominant_freq_feature_names = [f"{col}_dominant_freq" for col in eeg_columns]
    
    dominant_freq_df = pd.DataFrame(dominant_freq_features, columns=dominant_freq_feature_names)
    dominant_freq_df['target'] = targets
    
    output_file_path = '/Users/puchku-home/Downloads/Frequency Feature  Generalised/Dominant Frequency.csv'
    dominant_freq_df.to_csv(output_file_path, index=False)
    
    print(f"✅ Dominant Frequency feature extraction complete. Features saved to '{output_file_path}'")
